# Notebook 04 – Feature Engineering & Data Preprocessing

This notebook prepares the enriched banking dataset for machine learning.

Objectives:

- Select predictive features
- Encode categorical variables
- Engineer new features
- Scale numerical variables
- Split data into training and testing datasets
- Save preprocessing artifacts for future model training

The processed dataset generated in this notebook will be used in Notebook 05 for training multiple machine learning models.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
project_directory = Path.cwd().parent

data_directory = project_directory / "data"

processed_directory = data_directory / "processed"

processed_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_directory = project_directory / "models"

model_directory.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
enriched_dataset_path = (
    data_directory /
    "bank_customer_churn_enriched.csv"
)

df = pd.read_csv(
    enriched_dataset_path
)

print(
    f"Dataset Shape : {df.shape}"
)

df.head()

Dataset Shape : (10000, 23)


,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,Credit_Card_Status,...,Exited,Churn_Status,CreditScoreBand,AgeGroup,TenureGroup,BalanceStatus,BalanceSegment,HighValueCustomer,EngagementSegment,EDA_Risk_Segment
0,15634602,619,France,Female,42,2,0.00,1,1,Has Credit Card,...,1,Churned,Fair,41-50,New,Zero Balance,Zero Balance,Standard Value,Moderately Engaged,Higher Observed Risk
1,15647311,608,Spain,Female,41,1,83807.86,1,0,No Credit Card,...,0,Retained,Fair,41-50,New,Positive Balance,Low Balance,Standard Value,Moderately Engaged,Higher Observed Risk
2,15619304,502,France,Female,42,8,159660.80,3,1,Has Credit Card,...,1,Churned,Poor,41-50,Loyal,Positive Balance,Premium Balance,High Value,Moderately Engaged,Higher Observed Risk
3,15701354,699,France,Female,39,1,0.00,2,0,No Credit Card,...,0,Retained,Good,31-40,New,Zero Balance,Zero Balance,Standard Value,Moderately Engaged,Lower Observed Risk
4,15737888,850,Spain,Female,43,2,125510.82,1,1,Has Credit Card,...,0,Retained,Excellent,41-50,New,Positive Balance,High Balance,Standard Value,Moderately Engaged,Higher Observed Risk


In [4]:
print("="*70)
print("FEATURE ENGINEERING DATASET CHECK")
print("="*70)

print(f"Rows                : {len(df):,}")
print(f"Columns             : {len(df.columns)}")
print(f"Missing Values      : {df.isna().sum().sum()}")
print(f"Duplicate Customers : {df['CustomerId'].duplicated().sum()}")
print(f"Target Distribution :\n")
print(df["Exited"].value_counts())

print("\nTarget Percentage")

print(
    (
        df["Exited"]
        .value_counts(normalize=True)
        *100
    ).round(2)
)

FEATURE ENGINEERING DATASET CHECK
Rows                : 10,000
Columns             : 23
Missing Values      : 0
Duplicate Customers : 0
Target Distribution :

Exited
0    7963
1    2037
Name: count, dtype: int64

Target Percentage
Exited
0    79.63
1    20.37
Name: proportion, dtype: float64


In [6]:
columns_to_remove = [
    "CustomerId",
    "Surname",
    "Churn_Status",
    "Credit_Card_Status",
    "Activity_Status",
    "CreditScoreBand",
    "AgeGroup",
    "TenureGroup",
    "BalanceStatus",
    "BalanceSegment",
    "HighValueCustomer",
    "EngagementSegment",
    "EDA_Risk_Segment"
]

# Keep only columns that actually exist
existing_columns_to_remove = [
    column
    for column in columns_to_remove
    if column in df.columns
]

missing_columns = [
    column
    for column in columns_to_remove
    if column not in df.columns
]

df_ml = df.drop(
    columns=existing_columns_to_remove
).copy()

print("Columns removed:")
print(existing_columns_to_remove)

print("\nColumns already absent:")
print(missing_columns)

print("\nRemaining ML columns:")
print(df_ml.columns.tolist())

Columns removed:
['CustomerId', 'Churn_Status', 'Credit_Card_Status', 'Activity_Status', 'CreditScoreBand', 'AgeGroup', 'TenureGroup', 'BalanceStatus', 'BalanceSegment', 'HighValueCustomer', 'EngagementSegment', 'EDA_Risk_Segment']

Columns already absent:
['Surname']

Remaining ML columns:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


In [7]:
print("="*60)
print("FEATURES FOR MACHINE LEARNING")
print("="*60)

for i, column in enumerate(
    df_ml.columns,
    start=1
):
    print(f"{i:02d}. {column}")

FEATURES FOR MACHINE LEARNING
01. CreditScore
02. Geography
03. Gender
04. Age
05. Tenure
06. Balance
07. NumOfProducts
08. HasCrCard
09. IsActiveMember
10. EstimatedSalary
11. Exited


In [8]:
X = df_ml.drop(columns="Exited").copy()

y = df_ml["Exited"].copy()

print("Feature matrix shape :", X.shape)
print("Target vector shape  :", y.shape)

Feature matrix shape : (10000, 10)
Target vector shape  : (10000,)


In [9]:
categorical_features = [
    "Geography",
    "Gender"
]

numerical_features = [
    column
    for column in X.columns
    if column not in categorical_features
]

print("=" * 60)
print("Categorical Features")
print("=" * 60)
print(categorical_features)

print("\nNumerical Features")
print("=" * 60)
print(numerical_features)

Categorical Features
['Geography', 'Gender']

Numerical Features
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


In [10]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("Encoded feature matrix shape:", X_encoded.shape)

X_encoded.head()

Encoded feature matrix shape: (10000, 11)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,1,0


In [11]:
print("=" * 70)
print("ENCODED FEATURES")
print("=" * 70)

for i, column in enumerate(
    X_encoded.columns,
    start=1
):
    print(f"{i:02d}. {column}")

ENCODED FEATURES
01. CreditScore
02. Age
03. Tenure
04. Balance
05. NumOfProducts
06. HasCrCard
07. IsActiveMember
08. EstimatedSalary
09. Geography_Germany
10. Geography_Spain
11. Gender_Male


In [12]:
feature_names = list(X_encoded.columns)

feature_summary = pd.DataFrame({
    "Feature": feature_names
})

feature_summary.to_csv(
    processed_directory /
    "feature_list.csv",
    index=False
)

print("Feature list saved successfully.")

feature_summary

Feature list saved successfully.


,Feature
0,CreditScore
1,Age
2,Tenure
3,Balance
4,NumOfProducts
5,HasCrCard
6,IsActiveMember
7,EstimatedSalary
8,Geography_Germany
9,Geography_Spain


In [13]:
print("=" * 60)
print("ENCODING VERIFICATION")
print("=" * 60)

print("Missing values :", X_encoded.isna().sum().sum())

print("\nData types:\n")
print(X_encoded.dtypes)

print("\nShape :", X_encoded.shape)

ENCODING VERIFICATION
Missing values : 0

Data types:

CreditScore            int64
Age                    int64
Tenure                 int64
Balance              float64
NumOfProducts          int64
HasCrCard              int64
IsActiveMember         int64
EstimatedSalary      float64
Geography_Germany      int64
Geography_Spain        int64
Gender_Male            int64
dtype: object

Shape : (10000, 11)


In [14]:
X_encoded["BalanceSalaryRatio"] = (
    X_encoded["Balance"] /
    (X_encoded["EstimatedSalary"] + 1)
)

X_encoded["BalanceSalaryRatio"] = (
    X_encoded["BalanceSalaryRatio"]
    .round(4)
)

print("BalanceSalaryRatio created.")

BalanceSalaryRatio created.


In [15]:
X_encoded["ProductsPerYear"] = (
    X_encoded["NumOfProducts"] /
    (X_encoded["Tenure"] + 1)
)

X_encoded["ProductsPerYear"] = (
    X_encoded["ProductsPerYear"]
    .round(4)
)

print("ProductsPerYear created.")

ProductsPerYear created.


In [16]:
X_encoded["AgeTenureRatio"] = (
    X_encoded["Age"] /
    (X_encoded["Tenure"] + 1)
)

X_encoded["AgeTenureRatio"] = (
    X_encoded["AgeTenureRatio"]
    .round(4)
)

print("AgeTenureRatio created.")

AgeTenureRatio created.


In [17]:
X_encoded["ActiveBalance"] = (
    X_encoded["Balance"] *
    X_encoded["IsActiveMember"]
)

print("ActiveBalance created.")

ActiveBalance created.


In [18]:
X_encoded["ProductEngagementScore"] = (
    X_encoded["NumOfProducts"] *
    (X_encoded["IsActiveMember"] + 1)
)

print("ProductEngagementScore created.")

ProductEngagementScore created.


In [19]:
X_encoded["CustomerValueScore"] = (
    (
        X_encoded["Balance"] / 1000
    ) +
    (
        X_encoded["NumOfProducts"] * 10
    ) +
    (
        X_encoded["IsActiveMember"] * 20
    )
)

X_encoded["CustomerValueScore"] = (
    X_encoded["CustomerValueScore"]
    .round(2)
)

print("CustomerValueScore created.")

CustomerValueScore created.


In [20]:
engineered_features = [
    "BalanceSalaryRatio",
    "ProductsPerYear",
    "AgeTenureRatio",
    "ActiveBalance",
    "ProductEngagementScore",
    "CustomerValueScore"
]

print("=" * 65)
print("ENGINEERED FEATURES")
print("=" * 65)

for feature in engineered_features:
    print(feature)

print("\nTotal engineered features:",
      len(engineered_features))

ENGINEERED FEATURES
BalanceSalaryRatio
ProductsPerYear
AgeTenureRatio
ActiveBalance
ProductEngagementScore
CustomerValueScore

Total engineered features: 6


In [21]:
print("Dataset shape after feature engineering:")
print(X_encoded.shape)

X_encoded.head()

Dataset shape after feature engineering:
(10000, 17)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male,BalanceSalaryRatio,ProductsPerYear,AgeTenureRatio,ActiveBalance,ProductEngagementScore,CustomerValueScore
0,619,42,2,0.00,1,1,1,101348.88,0,0,0,0.0000,0.3333,14.0000,0.00,2,30.00
1,608,41,1,83807.86,1,0,1,112542.58,0,1,0,0.7447,0.5000,20.5000,83807.86,2,113.81
2,502,42,8,159660.80,3,1,0,113931.57,0,0,0,1.4014,0.3333,4.6667,0.00,3,189.66
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0.0000,1.0000,19.5000,0.00,2,20.00
4,850,43,2,125510.82,1,1,1,79084.10,0,1,0,1.5870,0.3333,14.3333,125510.82,2,155.51


In [22]:
all_features = list(X_encoded.columns)

pd.DataFrame({
    "Feature": all_features
}).to_csv(
    processed_directory /
    "engineered_feature_list.csv",
    index=False
)

print("Engineered feature list saved.")

print(f"Total features: {len(all_features)}")

Engineered feature list saved.
Total features: 17


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 65)
print("TRAIN TEST SPLIT")
print("=" * 65)

print(f"Training Features : {X_train.shape}")
print(f"Testing Features  : {X_test.shape}")

print(f"Training Labels   : {y_train.shape}")
print(f"Testing Labels    : {y_test.shape}")

TRAIN TEST SPLIT
Training Features : (8000, 17)
Testing Features  : (2000, 17)
Training Labels   : (8000,)
Testing Labels    : (2000,)


In [24]:
train_distribution = (
    y_train.value_counts(normalize=True)
    *100
).round(2)

test_distribution = (
    y_test.value_counts(normalize=True)
    *100
).round(2)

print("="*60)
print("TARGET DISTRIBUTION")
print("="*60)

print("\nTraining")
print(train_distribution)

print("\nTesting")
print(test_distribution)

TARGET DISTRIBUTION

Training
Exited
0    79.62
1    20.38
Name: proportion, dtype: float64

Testing
Exited
0    79.65
1    20.35
Name: proportion, dtype: float64


In [25]:
continuous_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "EstimatedSalary",
    "BalanceSalaryRatio",
    "ProductsPerYear",
    "AgeTenureRatio",
    "ActiveBalance",
    "CustomerValueScore"
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_features] = scaler.fit_transform(
    X_train[continuous_features]
)

X_test_scaled[continuous_features] = scaler.transform(
    X_test[continuous_features]
)

print("Scaling completed successfully.")

Scaling completed successfully.


In [26]:
verification = pd.DataFrame({
    "Mean": X_train_scaled[continuous_features].mean(),
    "Std": X_train_scaled[continuous_features].std()
}).round(3)

verification

,Mean,Std
CreditScore,-0.0,1.0
Age,0.0,1.0
Tenure,-0.0,1.0
Balance,0.0,1.0
EstimatedSalary,-0.0,1.0
BalanceSalaryRatio,-0.0,1.0
ProductsPerYear,-0.0,1.0
AgeTenureRatio,0.0,1.0
ActiveBalance,-0.0,1.0
CustomerValueScore,0.0,1.0


In [27]:
scaler_path = (
    model_directory /
    "standard_scaler.pkl"
)

joblib.dump(
    scaler,
    scaler_path
)

print(f"Scaler saved to:\n{scaler_path}")

Scaler saved to:
d:\Customer_Segmentation_Churn_Banking\models\standard_scaler.pkl


In [28]:
X_train_scaled.to_csv(
    processed_directory / "X_train.csv",
    index=False
)

X_test_scaled.to_csv(
    processed_directory / "X_test.csv",
    index=False
)

y_train.to_csv(
    processed_directory / "y_train.csv",
    index=False
)

y_test.to_csv(
    processed_directory / "y_test.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [29]:
print("=" * 70)
print("FEATURE ENGINEERING COMPLETED")
print("=" * 70)

print(f"Original Dataset        : {df.shape}")
print(f"ML Dataset             : {X_encoded.shape}")
print(f"Training Dataset       : {X_train_scaled.shape}")
print(f"Testing Dataset        : {X_test_scaled.shape}")

print(f"\nTotal Features         : {X_encoded.shape[1]}")
print(f"Engineered Features    : {len(engineered_features)}")

print(f"\nScaler Saved           : {scaler_path.exists()}")

print(
    f"Training Missing Values : "
    f"{X_train_scaled.isna().sum().sum()}"
)

print(
    f"Testing Missing Values  : "
    f"{X_test_scaled.isna().sum().sum()}"
)

print("\nNotebook 04 Completed Successfully!")

FEATURE ENGINEERING COMPLETED
Original Dataset        : (10000, 23)
ML Dataset             : (10000, 17)
Training Dataset       : (8000, 17)
Testing Dataset        : (2000, 17)

Total Features         : 17
Engineered Features    : 6

Scaler Saved           : True
Training Missing Values : 0
Testing Missing Values  : 0

Notebook 04 Completed Successfully!
